In [1]:
import numpy as np

In [2]:
with open("corpus.txt", "r") as file:
    words = [line.strip().lower() for line in file if line.strip()]

print("Corpus")
print(words)

Corpus
['low', 'lower', 'lowest']


In [3]:
characters = sorted(set("".join(words)))

characters.append("<EOS>")

char_to_index = {}
index_to_char = {}

for i, ch in enumerate(characters):
    char_to_index[ch] = i
    index_to_char[i] = ch

print("Vocabulary")
print(char_to_index)

Vocabulary
{'e': 0, 'l': 1, 'o': 2, 'r': 3, 's': 4, 't': 5, 'w': 6, '<EOS>': 7}


In [4]:
inputs = []
targets = []

for word in words:

    sequence = list(word)
    sequence.append("<EOS>")

    for i in range(len(sequence)-1):

        inputs.append(char_to_index[sequence[i]])
        targets.append(char_to_index[sequence[i+1]])

inputs = np.array(inputs)
targets = np.array(targets)

print("Inputs")
print(inputs)

print("\nTargets")
print(targets)

Inputs
[1 2 6 1 2 6 0 3 1 2 6 0 4 5]

Targets
[2 6 7 2 6 0 3 7 2 6 0 4 5 7]


In [5]:
embedding_size = 8

vocab_size = len(characters)

np.random.seed(7)

embeddings = np.random.randn(vocab_size, embedding_size)

embedded_inputs = embeddings[inputs]

print("Embedding Shape")
print(embedded_inputs.shape)

Embedding Shape
(14, 8)


In [6]:
sequence_length = len(embedded_inputs)

position = np.arange(sequence_length).reshape(-1,1)

dimension = np.arange(embedding_size).reshape(1,-1)

angle = position / np.power(10000, (2*(dimension//2))/embedding_size)

position_encoding = np.zeros((sequence_length, embedding_size))

position_encoding[:,0::2] = np.sin(angle[:,0::2])
position_encoding[:,1::2] = np.cos(angle[:,1::2])

embedded_inputs = embedded_inputs + position_encoding

print("Positional Encoding Shape")
print(position_encoding.shape)

Positional Encoding Shape
(14, 8)


In [7]:
d_model = embedding_size
d_k = 8

np.random.seed(10)

W_query = np.random.randn(d_model, d_k)
W_key = np.random.randn(d_model, d_k)
W_value = np.random.randn(d_model, d_k)

Query = embedded_inputs @ W_query
Key = embedded_inputs @ W_key
Value = embedded_inputs @ W_value

print("Query Shape :", Query.shape)
print("Key Shape   :", Key.shape)
print("Value Shape :", Value.shape)

Query Shape : (14, 8)
Key Shape   : (14, 8)
Value Shape : (14, 8)


In [8]:
scores = Query @ Key.T

scaled_scores = scores / np.sqrt(d_k)

print("Scaled Scores")
print(scaled_scores)

Scaled Scores
[[ -2.32514334  -3.62892214  -5.78635485 -10.20159233  -3.28894234
    2.75449024  -0.68412075  -5.03418269  -8.67824156  -6.95488268
   -2.05431644  -1.51357661  22.64198289  15.51322071]
 [-10.73299937 -11.22120103  24.83566026  -0.29852675  -5.20898825
   23.60923939  10.01601858  -9.42436926   1.50741221   0.53399226
   31.50592015  15.96152035  -4.96386536   5.27614299]
 [ -1.53896476  -8.08241574  18.38156582   5.11046357  -6.36765914
   16.13013436   7.75969377  -4.78586443   9.34653541   0.16847881
   22.3743754   11.10391693 -30.9016164  -13.54978948]
 [ -5.5687218   -4.66257128   3.21127567  -5.34298793  -0.73302665
    8.35707228  -0.20370902  -1.84860199  -4.60905697  -1.04320512
    9.15920522   2.79823956   7.32290655   3.89189905]
 [ -5.72897008  -6.02464151  22.98870468   6.47081974  -2.88301845
   15.99448985   9.35893468  -5.71603757   7.38473486   4.02953048
   24.60044939  13.5055695  -15.77958693  -3.87777299]
 [  8.15935109  -0.55462626   8.66662271 

In [9]:
mask = np.triu(np.ones((sequence_length, sequence_length)), k=1)

scaled_scores = np.where(mask==1, -1e9, scaled_scores)

print("Masked Scores")
print(scaled_scores)

Masked Scores
[[-2.32514334e+00 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09]
 [-1.07329994e+01 -1.12212010e+01 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09]
 [-1.53896476e+00 -8.08241574e+00  1.83815658e+01 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09]
 [-5.56872180e+00 -4.66257128e+00  3.21127567e+00 -5.34298793e+00
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09 -1.00000000e+09]
 [-5.72897008e+00 -6.02464151e+00  2.29887047e+01  6.4

In [10]:
def softmax(x):

    exp_values = np.exp(x - np.max(x, axis=-1, keepdims=True))

    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)

In [11]:
attention = softmax(scaled_scores)

print("Attention Weights")
print(attention)

Attention Weights
[[1.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [6.19682699e-01 3.80317301e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.23163667e-09 3.21246819e-12 9.99999998e-01 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [1.53666748e-04 3.80291020e-04 9.99273461e-01 1.92581636e-04
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [3.37343456e-13 2.50994279e-13 9.99999933e-01 6.70461260e-08
  5.80835805e-12 0.00000000e+00 0.00000000e+

In [12]:
context = attention @ Value

print("Context Shape")
print(context.shape)

print(context)

Context Shape
(14, 8)
[[ 4.38458836 -0.48268283  2.06340601  2.39606675 -1.72354929 -2.49012823
  -1.84760264 -4.40199776]
 [ 7.08958851 -0.02237667  1.21999036  1.70984783 -2.51560929 -1.17604213
  -2.12022811 -3.26815542]
 [ 5.08261364  4.1185239  -6.84713923 -2.68036259  0.36114118  5.8996648
   0.06695412 -3.22486564]
 [ 5.08436009  4.11573585 -6.84233435 -2.67796487  0.35893823  5.89546531
   0.06529502 -3.22436379]
 [ 5.08261344  4.11852364 -6.84713894 -2.68036247  0.36114108  5.89966446
   0.066954   -3.22486563]
 [ 3.26234822  1.05538678 -2.78458809 -0.75483022 -0.84922903  1.50613528
  -1.30800106 -3.42870744]
 [ 2.93367998  3.29000685 -5.29078871  0.33709398  1.44430316  4.57672426
   2.3448926  -2.75053213]
 [ 5.0812996   4.1180196  -6.84619359 -2.67851497  0.36180341  5.89886124
   0.0683544  -3.22457373]
 [ 2.94878827  3.30121283 -5.31296379  0.32130382  1.43815474  4.59676069
   2.34377633 -2.75042144]
 [ 5.08192126  4.11825874 -6.84664149 -2.67938861  0.36149066  5.89924

In [13]:
np.random.seed(20)

W_output = np.random.randn(d_k, vocab_size)

logits = context @ W_output

print("Logits Shape")
print(logits.shape)

print("\nLogits")
print(logits)

Logits Shape
(14, 8)

Logits
[[  0.93495695  -9.87930986  12.04418802   2.23007349  -2.2596373
   -7.98217753  -0.63910307  -9.15019634]
 [  2.00059132 -10.2690058   10.48223157  -2.67217773  -5.8602727
   -0.61031747   1.65532742  -9.5926388 ]
 [  6.64828758  -9.83617131  -2.83300926  -9.78481603  -1.32327268
   29.32784782  -4.73219415  -2.92963684]
 [  6.64489057  -9.83711577  -2.8254712   -9.78072549  -1.32683374
   29.31187228  -4.72744993  -2.9338529 ]
 [  6.6482871   -9.8361715   -2.83300889  -9.78481516  -1.32327249
   29.32784671  -4.73219407  -2.92963699]
 [  1.59689452 -11.38786757   2.98723731  -0.30228361   0.09186506
   13.19680005  -3.28690323  -5.27176556]
 [  7.6268471   -2.94743624  -2.35738292  -9.36952408  -0.15257094
   15.21443878  -4.63930986  -1.86871886]
 [  6.64889331  -9.83195357  -2.83272658  -9.78457478  -1.3225595
   29.31923346  -4.7321397   -2.92897973]
 [  7.63677461  -2.98020816  -2.37697806  -9.40024738  -0.1658783
   15.33825835  -4.64445959  -1.8603